### Bag of Stories
**Description:** Compute Kullback-Leibler divergence (KLD) in a set of stories. The code was created with Visual Studio Code (VSCode), GitHub Copilot support and Jupyter extension, under Windows 11.

**Contributor:** Florentina Armaselu  

In [ ]:
# Import required packages
from pathlib import Path
import numpy as np
from scipy.stats import entropy
from collections import Counter
import datetime as dt
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

Define the input, output paths.

In [ ]:
# Define the base directory relative to the current script location
base_dir = os.getcwd()
# Set the path to the input data folder for human stories
input = os.path.join(base_dir, "./../data/human/grimm/")
# Set the path to the input data folder for segmented human stories with text-tiling
input_seg_tt = os.path.join(base_dir, "./../data/annotated_stories/human/grimm/text-tiling/")
# Set the path to the input data folder for AI-generated stories with equal length start
input_ai_gen_el_st = os.path.join(base_dir, "./../data/ai-gen/grimm/el_start/")
# Set the path to the input data folder for AI-generated stories with text-tiling start
input_ai_gen_tt_st = os.path.join(base_dir, "./../data/ai-gen/grimm/tt_start/")
# Set the path to the input data folder for AI-generated stories with equal-length start and text-tiling segments
input_ai_gen_el_st_tt_seg = os.path.join(base_dir, "./../data/annotated_stories/ai-gen/grimm/el_start/text-tiling/")
# Set the path to the input data folder for AI-generated stories with text-tiling start and segments
input_ai_gen_tt_st_tt_seg = os.path.join(base_dir, "./../data/annotated_stories/ai-gen/grimm/tt_start/text-tiling/")
# Set the path to the output folder for KLD results on the human stories
output = os.path.join(base_dir, "./../results/human/kld/grimm/")
# Set the path to the output folder for KLD results on the AI-generated stories
output_ai_gen_el_st = os.path.join(base_dir, "./../results/ai-gen/kld/grimm/el_start/")
# Set the path of the output folder for KLD results on the AI-generated stories with text-tiling start
output_ai_gen_tt_st = os.path.join(base_dir, "./../results/ai-gen/kld/grimm/tt_start/")
# Set the path of the output folder for the intermediate results
output_inter_res = os.path.join(base_dir, "./../results/intermediate_results/grimm/")

Define constants.

In [ ]:
# String constants for KLD types
KLD_ADJACENT = 'adjacent'
KLD_CONTEXTUAL = 'contextual'

# String constants for story types
STORY_HUMAN = 'human'  # Label for human stories
STORY_AI_GEN = 'ai-gen'  # Label for AI-generated stories

# String constants for start types of AI-generated stories
GEN_EL_START = 'el_st' # Label for AI-generated stories with equal length start
GEN_TT_START = 'tt_st' # Label for AI-generated stories with text-tiling start

# String constants for segmentation methods
SEG_EQUAL_LENGTH = 'equal-length'
SEG_TEXT_TILING = 'text-tiling'

# String constants for annotation elements
ANN_SEG = 'Segment' # Label for segments
ANN_SEG_PAIR = 'Segment Pair' # Label for segment pairs
ANN_SEP = '-'*40 # Separator for segments

# String constants for output subfolders and file formats
OUT_CSV = 'csv'
OUT_PNG = 'png'

# String constants for file naming
FN_ADJACENT = 'adj'
FN_CONTEXTUAL = 'ctx'
FN_EQUAL_LENGTH = 'el'
FN_TEXT_TILING = 'tt'
FN_HUM = 'hum'
FN_GEN = 'gen'

# String constant for KLD labels
KLD_LABEL = 'KLD'
KLD_AVG = 'KLD Avg'
KLD_STD = 'KLD Std'
KLD_DEV = 'KLD Dev'

# String constants for types of intermediate results
INTER_RES_SEG = 'segment_check' # Intermediate results for segment check
INTER_RES_METH = 'method_check' # Intermediate results for method check

Define functions for pre-processing the text.

In [ ]:
def preprocess(text, rem_stopw=True, lemm=True):
    """
    Pre-processes the text.
    
    Args:
        text (str): Text to be pre-processed.
        rem_stopw (bool): Whether to remove stopwords
        lemm (bool): Whether to lemmatise
        
    Returns:
        list: A list of tokens.
    """   
    text = text.lower() # Convert to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    tokens = text.split()
    if rem_stopw:
        tokens = [t for t in tokens if t not in stopwords.words('english')]  # Remove stopwords
    if lemm:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(t) for t in tokens] # Lemmatise tokens
    return tokens

Define functions for story segmentation and reading of an annotated story.

In [ ]:
# Divide stories into segments of approximately equal length in number of tokens
def el_segment_story(story, seg_length=100):
    """
    Segments a story into smaller parts of approximately equal length in tokens.
    
    Args:
        story (str): The full text of the story.
        segment_length (int): The desired length of each segment in tokens.
        
    Returns:
        tuple: A tuple containing the character and token count of the story,
        and a list of segments, each containing approximately `segment_length` tokens.
    """   
    cnt_char = len(story) # Count the number of characters in the story
    tokens = story.split() # Split the story into tokens
    cnt_tok = len(tokens)  # Count the number of tokens in the story
    
    # Calculate the number of segments based on the desired segment length
    seg_count = math.ceil(len(tokens) / seg_length)
    
    # Length of each segment in tokens more equally distributed
    distrib_length = math.ceil(len(tokens) / seg_count) 
    
    # Create segments of approximately equal length
    # This will ensure that the segments are more evenly distributed in terms of token count
    segments = []
    for i in range(0, len(tokens), distrib_length):
        segment_tokens = tokens[i:i + distrib_length]
        segment = ' '.join(segment_tokens)
        if segment.strip():  # Add only non-empty segments
            segments.append(segment.strip())

    return cnt_char, cnt_tok, segments

# Read an annotated story.
def ann_segment_story(story):
    """
    Reads a text-tiling segmented story from a string and returns the segments.
    
    Args:
        story (str): The full text of the story.
        
    Returns:
        tuple: A tuple containing the character and token count of the story, and
        a list of segments extracted from the story.
    """
    cnt_char = 0  # Initialize the character count variable
    cnt_tok = 0  # Initialize the token count variable
    segments = [] # Initialize the list to hold segments

    # Split the story into parts based on the ANN_SEG label and ANN_SEP separator
    parts = story.split(ANN_SEP)
    num_sep = len(parts) - 1  # Count the number of ANN_SEP separators
    for part in parts:
        # Skip the first line if it contains the ANN_SEG label
        if part.strip().startswith(ANN_SEG):
            part = '\n'.join(part.split('\n')[2:])  # Remove the line containing the annotation label
        if part.strip():  # Add only non-empty segments
            cnt_char += len(part) # Count the number of characters in each segment and add to the total
            cnt_tok += len(part.strip().split()) # Count the number of tokens in each segment and add to the total
            segments.append(part.strip())
    
    cnt_char += num_sep # Add the number of separator lines to the character count
    return cnt_char, cnt_tok, segments

Define functions for computing Kullback-Leibler divergence in stories.

In [ ]:
# Compute Kullback-Leibler divergence for two segments
def compute_kld(segment1, segment2, save_inter_res, inter_res_meth_file):
    """
    Computes the Kullback-Leibler divergence between two segments.
    
    Args:
        segment1 (str): The first segment of text.
        segment2 (str): The second segment of text.
        save_inter_res (bool): Whether to save intermediate results to a file.
        inter_res_meth_file (str): File name for saving method check as intermediate results.  
        
    Returns:
        float: The Kullback-Leibler divergence between the two segments.
    """
    if segment1 == None or segment2 == None:
        raise ValueError("Cannot compute KL divergence with empty segments")
    
    # Pre-process and tokenise
    tokens1 = preprocess(segment1)
    tokens2 = preprocess(segment2)
    
    # Count the frequency of each token in both segments
    freq_list1 = Counter(tokens1)
    freq_list2 = Counter(tokens2)
    
    # Create a combined set of all unique tokens
    all_tokens = sorted(set(freq_list1.keys()).union(set(freq_list2.keys())))

    # Smoothing the probability distributions to avoid infinite KLD when 0 probabilities occur
    alpha = 0.5 # smoothing parameter (0.5 for Jeffreys)
    smooth_denom1 = len(tokens1) + alpha * len(all_tokens) # Smoothed denominator for segment 1
    smooth_denom2 = len(tokens2) + alpha * len(all_tokens) # Smoothed denominator for segment 2

   # Create smoothed probability distributions for both segments
    prob_dis1 = np.array([((freq_list1.get(token, 0) + alpha) / smooth_denom1) for token in all_tokens])
    prob_dis2 = np.array([((freq_list2.get(token, 0) + alpha) / smooth_denom2) for token in all_tokens])

    # Initialise contribution dictionary
    contributions = {}
    # Calculate contributions for each token, considering segment 2 as reference  
    for i, token in enumerate(all_tokens):
        p = prob_dis1[i]
        q = prob_dis2[i]
        contrib = q * math.log2(q / p) # Kullback-Leibler contribution of token i
        contributions[token] = contrib

    # If save_inter_res, save word contributions as intermediate results
    if save_inter_res:
        # Sort contributions to find the top and bottom 5 tokens
        sorted_contrib = sorted(contributions.items(), key=lambda x: x[1], reverse=True)
        top_5 = sorted_contrib[:5]
        bottom_5 = sorted_contrib[:-6:-1]
        # Sort the frequency lists
        dict_1 = sorted(dict(freq_list1).items(), key=lambda x: x[1], reverse=True)
        dict_2 = sorted(dict(freq_list2).items(), key=lambda x: x[1], reverse=True)

        with open(inter_res_meth_file, "a", encoding="utf-8") as f:
            f.write(f"{ANN_SEP[0:4]} Freq list 1: {dict_1}\n{ANN_SEP[0:4]} Freq list 2: {dict_2}\n")
            f.write(f"{ANN_SEP[0:4]} Top 5 KLD contributions: {top_5}\n{ANN_SEP[0:4]} Bottom 5 KLD contributions: {bottom_5}\n{ANN_SEP}\n")
    
    # Compute the Kullback-Leibler divergence
    kld = entropy(prob_dis2, prob_dis1, base=2) # Scipy KLD for segment 2 compared with segment 1
    
    return kld

# Compute Kullback-Leibler divergence for segments in a story
def compute_kld_for_story(story, kld_type='adjacent', seg_method='equal-length', save_inter_res=False, inter_res_seg_file=None, inter_res_meth_file=None):
    """
    Computes the Kullback-Leibler divergence for all segments in a story.
    
    Args:
        story (str): The full text of the story.
        kld_type (str): The type of KLD to compute ('adjacent' or 'contextual').
        seg_method (str): The method used for segmenting the stories ('equal-length', 'text-tiling', 'llm-episodes').
        save_inter_res (bool): Whether to save intermediate results to a file.
        inter_res_seg_file (str): File name for saving segment check as intermediate results.
        inter_res_meth_file (str): File name for saving method check as intermediate results.
        
    Returns:
        touple: A touple containing the character and token count of the story, and
        a list of Jensen-Shannon divergence values for each pair of segments.
    """
    # Segment the story into smaller parts
    cnt_char, cnt_tok, segments = ann_segment_story(story) if (seg_method == SEG_TEXT_TILING) else \
    el_segment_story(story, seg_length=100) 

    # Compute KLD for each pair of adjacent or contextual segments
    kld_values = []
    for i in range(len(segments) - 1):
        if kld_type == KLD_CONTEXTUAL:
            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEP[0:4]} {ANN_SEG} 0-{i}: {' '.join(segments[0:i+1])}\n{ANN_SEP[0:4]} {ANN_SEG} {i+1}: {segments[i + 1]}\n{ANN_SEP}\n")
                with open(inter_res_meth_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEG} 0-{i} - {ANN_SEG} {i+1}\n")
            # For contextual KLD, compute KLD between the next segment and all previous segments
            kld = compute_kld(' '.join(segments[0:i+1]), segments[i + 1], save_inter_res, inter_res_meth_file)
        else:
            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEP[0:4]} {ANN_SEG} {i}: {segments[i]}\n{ANN_SEP[0:4]} {ANN_SEG} {i+1}: {segments[i + 1]}\n{ANN_SEP}\n")
                with open(inter_res_meth_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEG} {i} - {ANN_SEG} {i+1}\n")
            # For adjacent KLD, compute KLD between the current segment and the next one  
            kld = compute_kld(segments[i], segments[i + 1], save_inter_res, inter_res_meth_file)
        kld_values.append(kld)
    
    return cnt_char, cnt_tok, kld_values

Define the function for analysing the stories and saving the results.

In [ ]:
# Process a set of stories to compute KLD values for story segments and save results as CSV and PNG files
def process_kld_stories(story_type=STORY_HUMAN, start_type=None, kld_type=KLD_ADJACENT, seg_method=SEG_EQUAL_LENGTH, save_inter_res=False):
    """
    Processes all stories in the input directory and saves the KLD results to the output directory.
    
    Args:
        story_type (str): The type of stories to process ('human' or 'ai-gen').
        start_type (str): The type of start for AI-generated stories ('gen_el_st' or 'gen_tt_st') or None for human stories.
        kld_type (str): The type of KLD to compute ('adjacent' or 'contextual').
        seg_method (str): The method used for segmenting the stories ('equal-length', 'text-tiling', 'llm-episodes').
        save_inter_res (bool): Whether to save intermediate results to a file.
    """
    cnt_processed = 0 # Counter for processed files
    kld_avg_values = [] # List to store KLD averages for each story
    inp_file_names = [] # List to store input file names
    story_names = [] # List of story names
    num_chars = [] # Counter list for the number of characters processed
    num_toks = [] # Counter list for the number of tokens processed
    num_segs = [] # Counter list for the number of segments processed
    fn_kld_type=FN_ADJACENT # File name suffix for KLD type, default is 'adjacent'
    fn_seg_meth=FN_EQUAL_LENGTH # File name suffix for segmentation method, default is 'equal-length'
    fn_story = FN_HUM # File name suffix for human stories, default is 'hum'
    fn_source = "" # File name suffix for source stories, default is empty
    inp = input # Default input path for whole stories
    out = output # Default output path for human stories

# Select the input path and the file name suffix based on the story type and start type
    if story_type == STORY_HUMAN:
        start_type = "none"  # No start type for human stories
        if seg_method == SEG_TEXT_TILING:
            inp = input_seg_tt
    elif story_type == STORY_AI_GEN:
        fn_story = FN_GEN  # Set the file name suffix for AI-generated stories
        if start_type == GEN_EL_START:
            fn_source = GEN_EL_START + "_" # Source for AI-generated stories with equal length start
            if seg_method == SEG_TEXT_TILING:
                inp = input_ai_gen_el_st_tt_seg
            else:
                inp = input_ai_gen_el_st
            out = output_ai_gen_el_st
        elif start_type == GEN_TT_START:
            fn_source = GEN_TT_START + "_" # Source for AI-generated stories with text-tiling start
            if seg_method == SEG_TEXT_TILING:
                inp = input_ai_gen_tt_st_tt_seg
            else:
                inp = input_ai_gen_tt_st
            out = output_ai_gen_tt_st

    # Select the file name suffix based on the segmentation method
    if seg_method == SEG_TEXT_TILING:
        fn_seg_meth = FN_TEXT_TILING

    # Select the file name suffix based on the KLD type
    if kld_type == KLD_CONTEXTUAL:
        fn_kld_type = FN_CONTEXTUAL

    timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')  # Create a timestamp for the output files

    # File names for intermediate results
    inter_res_seg_file = None # Segment check file
    inter_res_meth_file = None # Probability distribution file 
    if save_inter_res: # Create the intermediate results files if saving is enabled
        inter_res_seg_file = os.path.join(output_inter_res, f"{INTER_RES_SEG}_{fn_source}{fn_kld_type}_{fn_seg_meth}_{KLD_LABEL.lower()}_{fn_story}_{timestamp}.txt")
        with open(inter_res_seg_file, "w", encoding="utf-8") as f:
            f.write(f"{INTER_RES_SEG.replace('_', ' ').capitalize()}: Kullback-Leibler Divergence (KLD): story type: {story_type}; story start: {start_type}; KLD type: {kld_type}; segmentation method: {seg_method}\n{ANN_SEP}\n")
        inter_res_meth_file = os.path.join(output_inter_res, f"{INTER_RES_METH}_{fn_source}{fn_kld_type}_{fn_seg_meth}_{KLD_LABEL.lower()}_{fn_story}_{timestamp}.txt")
        with open(inter_res_meth_file, "w", encoding="utf-8") as f:
            f.write(f"{INTER_RES_METH.replace('_', ' ').capitalize()}: Kullback-Leibler (KLD): story type: {story_type}; story start: {start_type}; KLD type: {kld_type}; segmentation method: {seg_method}\n{ANN_SEP}\n")

    # Read the story files sequentially from the input folder.
    for file_path in Path(inp).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as input_file:
            story = input_file.read()
    
            # Print the name of the file being processed.
            print(f'Processing file: {file_path.name}')
            inp_file_names.append(file_path.name)  # Store the input file name

            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"Processing file: {file_path.name}\n{ANN_SEP}\n")
                with open(inter_res_meth_file, "a", encoding="utf-8") as f:
                    f.write(f"Processing file: {file_path.name}\n{ANN_SEP}\n")

            if (seg_method == SEG_TEXT_TILING):
                # Get the title of the story
                start = story.find(ANN_SEG + " 0:\n") + len(ANN_SEG + " 0:\n")
                end = story.find(ANN_SEP)
                story_name = story[start:end].strip()
                # Get the body of the story, excluding the title
                body = story[end+len(ANN_SEP):]
            else:   
                # Get the title of the story
                story_name = story.split('\n')[0]
                # Get the body of the story, excluding the title
                body = story[story.find('\n')+1:]
                
            # Compute KLD values for the story
            cnt_char, cnt_tok, kld_values = compute_kld_for_story(body, kld_type, seg_method, save_inter_res, inter_res_seg_file, inter_res_meth_file)
            
            kld_avg = np.mean(kld_values)  # KLD average for the story
            kld_avg_values.append(kld_avg)  # Store the KLD average for the story
            story_names.append(story_name)
            num_chars.append(cnt_char)  # Store the number of characters in the story
            num_toks.append(cnt_tok)  # Store the number of tokens in the story
            num_segs.append(len(kld_values)+1)  # Store the number of segments in the story

            # Story KLD table
            df = pd.DataFrame({
                ANN_SEG_PAIR : range(1, len(kld_values) + 1),
                KLD_LABEL : kld_values,
                KLD_DEV : [kld_val - kld_avg for kld_val in kld_values],  # KLD deviation from the average by segment pair
                KLD_AVG : kld_avg, # KLD average for the story
                KLD_STD : np.std(kld_values)  # Standard deviation of KLD for the story
            })
            
            # Clean the file_path name of timestamps, to shorten the file name
            clean_stem = re.sub(r'\d{8}_\d{6}', '#', file_path.stem)
            
            # Save the story DataFrame to a CSV file
            csv_path = os.path.join(out, OUT_CSV, f"{clean_stem}_{fn_kld_type}_{fn_seg_meth}_{KLD_LABEL.lower()}_{fn_story}_{timestamp}.{OUT_CSV}")
            df.to_csv(csv_path, index=False)

            # Save the line plot as PNG
            plt.figure(figsize=(8, 4))
            plt.plot(df[ANN_SEG_PAIR], df[KLD_LABEL], marker='o')
            plt.xlabel(ANN_SEG_PAIR)
            plt.ylabel('Kullback-Leibler Divergence')
            plt.title(f'Surprisal (KLD) across {kld_type} {seg_method.replace('llm', 'LLM')} segments, {story_type} story, story start ({start_type}): {story_name}', wrap=True, y = 1.02)
            plt.grid(True)
            plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))  # Ensure integer ticks
            plt.xlim(left=0)  # Start x-axis from 0
            # Set x-ticks to include the first and last segment numbers
            ticks = np.unique(np.concatenate(([df[ANN_SEG_PAIR].min()], plt.gca().get_xticks(), [df[ANN_SEG_PAIR].max()])))
            plt.xticks(ticks.astype(int))  # Set ticks (as integers)
            plt.tight_layout(rect=[0, 0, 1, 0.98])
            png_path = os.path.join(out, OUT_PNG, f"{clean_stem}_{fn_kld_type}_{fn_seg_meth}_{KLD_LABEL.lower()}_{fn_story}_{timestamp}.{OUT_PNG}")
            plt.savefig(png_path)
            plt.close()

            cnt_processed += 1

    kld_avg_by_run = np.mean(kld_avg_values)  # Average KLD accross the run

    # Summary DataFrame for the run
    df_avg = pd.DataFrame({
        'ID': range(1, len(kld_avg_values) + 1),
        'File': inp_file_names,  # Input file names 
        'Story': story_names,
        'Characters': num_chars,  # Number of characters by story
        'Tokens': num_toks,  # Number of tokens by story  
        'Segments': num_segs,  # Number of segments by story
        KLD_AVG: kld_avg_values,  # KLD average by story
        KLD_DEV: [kld_val - kld_avg_by_run for kld_val in kld_avg_values],  # KLD deviation from the average by story
        KLD_AVG + ' Run': kld_avg_by_run, # KLD average by run
        KLD_STD + ' Run': np.std(kld_avg_values)  # Standard deviation of KLD by run
    })

    # Save the summary DataFrame to a CSV file
    csv_avg_path = os.path.join(out, OUT_CSV, f"summary_{fn_source}{fn_kld_type}_{fn_seg_meth}_{KLD_LABEL.lower()}_{fn_story}_{timestamp}.{OUT_CSV}")
    df_avg.to_csv(csv_avg_path, index=False)
    
    # Print the number of files saved in the output folder and the summary file name.
    end_proc_str = f'End processing: {cnt_processed} files saved to the {Path(out).name} folder.'
    print(end_proc_str)
    if save_inter_res:
        with open(inter_res_seg_file, "a", encoding="utf-8") as f:
            f.write(f"{end_proc_str}\n")
        with open(inter_res_meth_file, "a", encoding="utf-8") as f:
            f.write(f"{end_proc_str}\n")

    print(f'Summary of KLD values across the run saved to the {Path(csv_avg_path).name} file.')

Main processing: validate input, read the stories, analyse them, and save the results.  

In [ ]:
# Choose the KLD and story type and segmentation method
story_type = STORY_HUMAN # Choose between STORY_HUMAN or STORY_AI_GEN
start_type = None # Choose between GEN_EL_START or GEN_TT_START for AI-generated stories or None for human stories
kld_type = KLD_ADJACENT # KLD_ADJACENT or KLD_CONTEXTUAL
seg_method = SEG_EQUAL_LENGTH  # SEG_EQUAL_LENGTH or SEG_TEXT_TILING

# Set the flag to save intermediate results
save_inter_res = False  # Set to True to save intermediate results for segment and method check, False otherwise

# Validate the story type
if story_type not in [STORY_HUMAN, STORY_AI_GEN]:
    raise ValueError("Invalid story type. Choose from ", STORY_HUMAN, "or ", STORY_AI_GEN, ".")

# Validate the start type for AI-generated stories
if story_type == STORY_AI_GEN and start_type not in [GEN_EL_START, GEN_TT_START]:
    raise ValueError("Invalid start type for AI-generated stories. Choose from ", GEN_EL_START, "or ", GEN_TT_START, ".")

# Validate the KLD type
if kld_type not in [KLD_ADJACENT, KLD_CONTEXTUAL]:
    raise ValueError("Invalid KLD type. Choose from ", KLD_ADJACENT, "or ", KLD_CONTEXTUAL, ".")

# Validate the segmentation method and set the input path accordingly
if seg_method not in [SEG_EQUAL_LENGTH, SEG_TEXT_TILING]:
    raise ValueError("Invalid segmentation method. Choose from ", SEG_EQUAL_LENGTH, ", ", SEG_TEXT_TILING, ".")

# Process the stories with the specified story and KLD type, segmentation method and flag for saving intermediate results
process_kld_stories(story_type, start_type, kld_type, seg_method, save_inter_res)